# 05 · Evaluation: Official Scoring and Safety-Stock Dollars

**Owner:** Elena · **Reads:** every forecast file in `data/forecasts/` · **Feeds:** results tables and charts for the blog and slides

This is the one place every model gets scored the same way. It picks up whatever forecast files exist, so re-run it whenever a teammate adds a model.

**What it reports**
1. **Mean absolute error (MAE):** average units off per product per day. Lower is better.
2. **Sales-weighted MAE:** the same, but products that bring in more revenue count more (M5-style weighting by dollar sales in the 28 days before each window).
3. **Bias:** do forecasts run high (+) or low (−) on average? Matters for inventory.
4. **Breakdowns** by category (Foods, Hobbies, Household) and sales speed (fast, medium, slow).
5. **Safety-stock dollars:** how much buffer inventory each model needs for a 95% service level, and whether that buffer actually covers 95% of weeks on the test window.

**Fairness rule:** within each window, every model is scored on exactly the same products and days (the ones all models forecast).

**Runtime:** CPU is fine. A few minutes.

## 0 · Settings

In [ ]:
SAMPLE   = False     # True = score the 100-product smoke-test forecasts instead
WINDOWS  = ["val_1", "val_2", "test"]          # windows to report (backtests are used by calibration later)
SERVICE_Z   = 1.65   # 95% service level
LEAD_DAYS   = 7      # restocking lead time assumed for safety stock
DRIVE_DIR   = None   # None = find AML_Term_Project in My Drive automatically; or paste a path

## 1 · Connect and find the shared folder

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set DRIVE_DIR in Settings to your local AML_Term_Project folder.")

if DRIVE_DIR is None:
    found = (glob.glob("/content/drive/MyDrive/AML_Term_Project")
             or glob.glob("/content/drive/MyDrive/*/AML_Term_Project"))
    assert found, "Can't find AML_Term_Project in My Drive. Add the shortcut, or set DRIVE_DIR."
    DRIVE_DIR = found[0]
DATA_DIR = f"{DRIVE_DIR}/data"
FC_DIR   = f"{DATA_DIR}/forecasts_sample" if SAMPLE else f"{DATA_DIR}/forecasts"
OUT_DIR  = f"{DATA_DIR}/results_sample" if SAMPLE else f"{DATA_DIR}/results"
FIG_DIR  = f"{DATA_DIR}/figures"
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print("Reading forecasts from", FC_DIR)

## 2 · Load every forecast file
Each file must have `unique_id`, `ds`, `window` and `forecast` (and ideally `model`). Extra columns are ignored.
Model names are cleaned to lowercase so `DeepAR` and `deepar` count as one model.

In [ ]:
frames, skipped = [], []
for path in sorted(glob.glob(f"{FC_DIR}/*.parquet")):
    f = pd.read_parquet(path)
    need = {"unique_id", "ds", "window", "forecast"}
    if not need.issubset(f.columns):
        skipped.append((os.path.basename(path), sorted(need - set(f.columns)))); continue
    if "model" not in f.columns:
        f["model"] = os.path.basename(path).rsplit("_", 1)[0]
    f = f[["unique_id", "ds", "model", "window", "forecast"]].copy()
    f["model"] = f["model"].astype(str).str.lower()
    f["unique_id"] = f["unique_id"].astype(str)
    f["ds"] = pd.to_datetime(f["ds"])
    f["forecast"] = f["forecast"].astype("float64").clip(lower=0)
    frames.append(f)

fc = pd.concat(frames, ignore_index=True)
fc = fc.drop_duplicates(["unique_id", "ds", "model", "window"], keep="last")
if skipped:
    print("Skipped (missing columns):", skipped)
print(f"{len(frames)} files, {fc['model'].nunique()} models:", sorted(fc["model"].unique()))
fc.groupby(["model", "window"])["unique_id"].nunique().unstack().reindex(columns=[w for w in
    ["bt_4", "bt_3", "bt_2", "bt_1", "val_1", "val_2", "test"] if w in fc["window"].unique()])

## 3 · Attach actual sales, prices, category and sales speed

In [ ]:
sales = pd.read_parquet(f"{DATA_DIR}/{'ca_sample100' if SAMPLE else 'ca_sales'}.parquet",
                        columns=["unique_id", "ds", "y", "sell_price"])
sales["unique_id"] = sales["unique_id"].astype(str)
sales["ds"] = pd.to_datetime(sales["ds"])
static = pd.read_parquet(f"{DATA_DIR}/ca_static.parquet")[["unique_id", "cat_id", "sales_speed"]]
static["unique_id"] = static["unique_id"].astype(str)
with open(f"{DATA_DIR}/windows.json") as fh:
    wjson = json.load(fh)
H = wjson["horizon"]

df = (fc[fc["window"].isin(WINDOWS)]
      .merge(sales, on=["unique_id", "ds"], how="inner")
      .merge(static, on="unique_id", how="left"))
df["err"] = df["forecast"] - df["y"]
df["abs_err"] = df["err"].abs()
print(f"{len(df):,} scored rows")

## 4 · Fair comparison set
Within each window, keep only the product-days that **every** model forecast. A model that skipped some products
(for example, ones launched recently) can't look better by dodging hard cases.

In [ ]:
models_by_window = df.groupby("window")["model"].nunique()
keep = (df.groupby(["window", "unique_id", "ds"])["model"].transform("nunique")
        == df["window"].map(models_by_window))
dropped = 1 - keep.mean()
df = df[keep].copy()
print(f"Dropped {dropped:.1%} of rows not covered by every model in their window")
df.groupby("window").agg(models=("model", "nunique"), products=("unique_id", "nunique"))

## 5 · Sales weights (M5-style)
Each product's weight in a window = its dollar sales in the 28 days **before** that window, as a share of the total.
High-revenue products count more, which is closer to what a retailer cares about.

In [ ]:
wrows = []
for w in WINDOWS:
    if w not in wjson: continue
    start = pd.Timestamp(wjson[w]["start"])
    prior = sales[(sales["ds"] < start) & (sales["ds"] >= start - pd.Timedelta(days=H))]
    dollars = (prior["y"] * prior["sell_price"]).groupby(prior["unique_id"]).sum()
    wrows.append(pd.DataFrame({"window": w, "unique_id": dollars.index, "weight": (dollars / dollars.sum()).values}))
weights = pd.concat(wrows, ignore_index=True)
df = df.merge(weights, on=["window", "unique_id"], how="left")
df["weight"] = df["weight"].fillna(0)

## 6 · Headline results: every model, every window

In [ ]:
def score(g):
    per_series = g.groupby("unique_id").agg(mae=("abs_err", "mean"), w=("weight", "first"))
    wsum = per_series["w"].sum()
    return pd.Series({
        "MAE": g["abs_err"].mean(),
        "Weighted MAE": (per_series["mae"] * per_series["w"]).sum() / wsum if wsum > 0 else np.nan,
        "Bias": g["err"].mean(),
        "Products": g["unique_id"].nunique(),
    })

overall = df.groupby(["window", "model"])[["unique_id", "abs_err", "err", "weight"]].apply(score).reset_index()
overall.to_csv(f"{OUT_DIR}/overall_by_window.csv", index=False)

mae_table = overall.pivot(index="model", columns="window", values="MAE")[[w for w in WINDOWS if w in overall["window"].unique()]]
mae_table = mae_table.sort_values(mae_table.columns[-1])
print("MAE (units per product per day; lower is better)")
mae_table.round(3)

In [ ]:
wmae_table = overall.pivot(index="model", columns="window", values="Weighted MAE")[mae_table.columns]
print("Sales-weighted MAE")
wmae_table.loc[mae_table.index].round(3)

In [ ]:
bias_table = overall.pivot(index="model", columns="window", values="Bias")[mae_table.columns]
print("Bias (average forecast minus actual; negative = forecasts run low)")
bias_table.loc[mae_table.index].round(3)

### Improvement over the baselines (test window)
Percent less error than each baseline. `smean4` (average of the last 4 same weekdays) is the stronger, more honest baseline to quote.

In [ ]:
test = overall[overall["window"] == "test"].set_index("model")
rows = []
for base in [b for b in ["snaive", "smean4"] if b in test.index]:
    for m in test.index:
        rows.append({"model": m, "vs": base,
                     "MAE improvement": 1 - test.loc[m, "MAE"] / test.loc[base, "MAE"],
                     "Weighted MAE improvement": 1 - test.loc[m, "Weighted MAE"] / test.loc[base, "Weighted MAE"]})
imp = pd.DataFrame(rows)
imp.to_csv(f"{OUT_DIR}/improvement_vs_baselines_test.csv", index=False)
(imp.pivot(index="model", columns="vs", values="MAE improvement")
    .loc[mae_table.index].map(lambda v: f"{v:+.0%}"))

## 7 · Where each model wins (test window)

In [ ]:
t = df[df["window"] == "test"]
by_cat = t.groupby(["cat_id", "model"])["abs_err"].mean().unstack().T.loc[mae_table.index]
by_speed = t.groupby(["sales_speed", "model"], observed=True)["abs_err"].mean().unstack().T.loc[mae_table.index]
by_speed = by_speed[[c for c in ["fast", "medium", "slow"] if c in by_speed.columns]]
by_cat.to_csv(f"{OUT_DIR}/test_mae_by_category.csv"); by_speed.to_csv(f"{OUT_DIR}/test_mae_by_speed.csv")
print("MAE by category"); display(by_cat.round(3))
print("MAE by sales speed"); display(by_speed.round(3))

## 8 · Safety-stock dollars
For each product and model:
1. **Error spread** = standard deviation of that model's daily errors on the two validation windows (56 days). The test window is never used to size the buffer.
2. **Safety stock (units)** = 1.65 × spread × √7 (95% service, 7-day lead time).
3. **Dollars** = units × the product's average price on the test window.

Then a reality check on the test window: for each product and each of its 4 test weeks, is
**actual weekly demand ≤ forecast for the week + safety stock**? A well-sized buffer should cover about 95% of weeks.

In [ ]:
val = df[df["window"].isin(["val_1", "val_2"])]
test_rows = df[df["window"] == "test"].copy()
spread = val.groupby(["model", "unique_id"])["err"].std().rename("sigma").reset_index()
price = (test_rows.groupby("unique_id")["sell_price"].mean().rename("price").reset_index())

ss = spread.merge(price, on="unique_id", how="inner")
ss["ss_units"] = SERVICE_Z * ss["sigma"] * np.sqrt(LEAD_DAYS)
ss["ss_dollars"] = ss["ss_units"] * ss["price"]

test_rows["week"] = ((test_rows["ds"] - test_rows["ds"].min()).dt.days // 7)
weekly = test_rows.groupby(["model", "unique_id", "week"]).agg(demand=("y", "sum"), fcst=("forecast", "sum")).reset_index()
weekly = weekly.merge(ss[["model", "unique_id", "ss_units"]], on=["model", "unique_id"], how="inner")
weekly["covered"] = weekly["demand"] <= weekly["fcst"] + weekly["ss_units"]

ss_summary = (ss.groupby("model").agg(**{"Safety stock (units)": ("ss_units", "sum"),
                                         "Safety stock ($)": ("ss_dollars", "sum")})
              .join(weekly.groupby("model")["covered"].mean().rename("Weeks covered (target 95%)")))
best_base = "smean4" if "smean4" in ss_summary.index else ("snaive" if "snaive" in ss_summary.index else None)
if best_base:
    ss_summary[f"$ saved vs {best_base}"] = ss_summary.loc[best_base, "Safety stock ($)"] - ss_summary["Safety stock ($)"]
ss_summary = ss_summary.sort_values("Safety stock ($)")
ss_summary.to_csv(f"{OUT_DIR}/safety_stock_test.csv")
fmt = {"Safety stock (units)": "{:,.0f}", "Safety stock ($)": "${:,.0f}", "Weeks covered (target 95%)": "{:.1%}"}
if best_base: fmt[f"$ saved vs {best_base}"] = "${:,.0f}"
shown = ss_summary.copy()
for col, f in fmt.items():
    shown[col] = shown[col].map(f.format)
shown

**How to read it:** a model with a smaller dollar buffer *and* coverage near 95% is the better inventory forecaster.
If a model's coverage is far below 95%, its buffer is too small to trust; far above, it is wasting money.

## 9 · Charts for the blog and slides

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
LABELS = {"snaive": "Naive (last week)", "smean4": "Naive (4-wk avg)", "chronos2_zs": "Chronos-2",
          "chronos2_cov": "Chronos-2 + covariates", "timesfm3_zs": "TimesFM-3", "timesfm3_cov": "TimesFM-3 + covariates",
          "deepar": "DeepAR", "lgbm": "LightGBM", "lgbm_cov": "LightGBM + covariates"}
lab = lambda m: LABELS.get(m, m)
BASE = {"snaive", "smean4"}

order = mae_table["test"].sort_values(ascending=False) if "test" in mae_table else mae_table.iloc[:, -1].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 0.45 * len(order) + 1.2))
colors = ["#A9B1BE" if m in BASE else "#BF5700" for m in order.index]
ax.barh([lab(m) for m in order.index], order.values, color=colors)
for i, v in enumerate(order.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("Mean absolute error, test window (units per product per day)")
ax.set_title("Test-window error by model (lower is better)")
ax.set_xlim(0, order.max() * 1.15)
plt.tight_layout()
if not SAMPLE: plt.savefig(f"{FIG_DIR}/20_test_mae_by_model.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
speed_plot = by_speed.loc[[m for m in order.index[::-1]]]
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(speed_plot.columns)); n = len(speed_plot)
width = 0.8 / n
for i, (m, row) in enumerate(speed_plot.iterrows()):
    ax.bar(x + i * width - 0.4 + width / 2, row.values, width, label=lab(m),
           color=None if m not in BASE else "#A9B1BE")
ax.set_xticks(x); ax.set_xticklabels([c.capitalize() + " sellers" for c in speed_plot.columns])
ax.set_ylabel("MAE, test window"); ax.set_title("Error by sales speed")
ax.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
if not SAMPLE: plt.savefig(f"{FIG_DIR}/21_test_mae_by_speed.png", dpi=150, bbox_inches="tight")
plt.show()

## What to report back
Results tables are saved as CSVs in `data/results/` and charts in `data/figures/`.
Download this notebook with outputs (File → Download .ipynb) and send it to Max, or commit it to `notebooks/05_evaluation.ipynb`.

**Questions to answer in the blog:**
1. Which model has the lowest test error, and by how much does it beat the 4-week-average baseline?
2. Does the ranking change between fast and slow sellers?
3. Are forecasts biased low? (Expected for the foundation models' medians.)
4. Which model needs the least safety stock while still covering about 95% of weeks?